# Oil Price Shocks and Texas Employment

## ETL Pipeline (Data Collection and Cleaning)
**Group:** Anthony Bauer, Jackson Daniel, Logan Averill

This notebook performs the data collection and cleaning steps for the project.

### Research Question
Do changes in WTI oil prices predict county-level employment growth in Texas energy-related industries, and does this relationship differ after major oil price shocks? (2010-2025)

### Data Sources
- FRED: WTI Oil Prices (WTISPLC), Texas Unemployment Rate
- BLS QCEW: County-level employment by industry (NAICS codes 211 and 213)

### Workflow
1. Load raw datasets
2. Filter for Texas counties
3. Extract energy sector employment
4. Prepare data for merging
5. Output cleaned dataset for analysis

All steps are written to be fully reproducible.

# 0) Imports + File Paths 

In [9]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [10]:
# File paths

oil_path = "../data_raw/WTISPLC.csv"
unemp_path = "../data_raw/TXUR.csv"

print("Done.")

Done.


# 1) Load and Clean Data

## Most Efficient Loading of QCEW Data

A single QCEW dataset is extremely large. So, with 15 datasets with over 10 million rows and 42 variables, attempting to load the full datasets all at once was basically impossible with slow performance and high memory usage.

To address this and to actually be able to use this data, a kind of strange loading approach was implemented using chunking and early filtering. 

### Key Steps
- **Column Selection**: Only the necessary columns will be loaded (`area_fips`,`industry_code`(`area_fips`, `industry_code`, `year`, `qtr`, and employment variables), reducing memory usage.
- **Chunk Processing**: The dataset will be read in small chunks rather than all of them at once, which will allow the code to process large files much easier.
- **Early Filtering**:
  - The data will be restricted to Texas counties using FIPS codes beginning with 48.
  - The data will be filtered for energy-related industries (NAICS 211 and 213).

In [12]:
# Load datasets
oil = pd.read_csv(oil_path)
unemp = pd.read_csv(unemp_path)
# Load and combine all QCEW files
# Faster QCEW loading: only keep needed columns and filter while reading

cols_needed = [
    "area_fips",
    "industry_code",
    "year",
    "qtr",
    "month1_emplvl",
    "month2_emplvl",
    "month3_emplvl"
]

qcew_files = [
    "2025.q1-q3.singlefile.csv",
    "2024.q1-q4.singlefile.csv",
    "2023.q1-q4.singlefile.csv",
    "2022.q1-q4.singlefile.csv",
    "2021.q1-q4.singlefile.csv",
    "2020.q1-q4.singlefile.csv",
    "2019.q1-q4.singlefile.csv",
    "2018.q1-q4.singlefile.csv",
    "2017.q1-q4.singlefile.csv",
    "2016.q1-q4.singlefile.csv",
    "2015.q1-q4.singlefile.csv",
    "2014.q1-q4.singlefile.csv",
    "2013.q1-q4.singlefile.csv",
    "2012.q1-q4.singlefile.csv",
    "2011.q1-q4.singlefile.csv",
    "2010.q1-q4.singlefile.csv",
]

filtered_chunks = []

for file in qcew_files:
    print(f"Processing {file}...")

    chunks = pd.read_csv(
        f"../data_raw/{file}",
        usecols=cols_needed,
        dtype={"area_fips": str, "industry_code": str},
        chunksize=500_000,
        low_memory=False
    )

    for chunk in chunks:
        chunk["area_fips"] = chunk["area_fips"].str.zfill(5)

        # Texas only
        chunk = chunk[chunk["area_fips"].str.startswith("48")]

        # Energy industries only
        chunk = chunk[chunk["industry_code"].str.startswith(("211", "213"))]

        filtered_chunks.append(chunk)

qcew = pd.concat(filtered_chunks, ignore_index=True)



print("Data loaded.\n")

print("Filtered QCEW shape:", qcew.shape)
print("Oil shape:", oil.shape)
print("Unemployment shape:", unemp.shape)

Processing 2025.q1-q3.singlefile.csv...
Processing 2024.q1-q4.singlefile.csv...
Processing 2023.q1-q4.singlefile.csv...
Processing 2022.q1-q4.singlefile.csv...
Processing 2021.q1-q4.singlefile.csv...
Processing 2020.q1-q4.singlefile.csv...
Processing 2019.q1-q4.singlefile.csv...
Processing 2018.q1-q4.singlefile.csv...
Processing 2017.q1-q4.singlefile.csv...
Processing 2016.q1-q4.singlefile.csv...
Processing 2015.q1-q4.singlefile.csv...
Processing 2014.q1-q4.singlefile.csv...
Processing 2013.q1-q4.singlefile.csv...
Processing 2012.q1-q4.singlefile.csv...
Processing 2011.q1-q4.singlefile.csv...
Processing 2010.q1-q4.singlefile.csv...
Data loaded.

Filtered QCEW shape: (127977, 7)
Oil shape: (963, 2)
Unemployment shape: (601, 2)


## Constructing Employment Measures and Panel Structure

After filtering the QCEW dataset to include only relevant observations, several steps will be taken to prepare the data for analysis.

### Quarterly Employment

The QCEW dataset provides employment levels for each of the three months within a quarter. To create a single measure of employment per quarter, the average of the three monthly values will be calculated:

- `month1_emplvl`
- `month2_emplvl`
- `month3_emplvl`

This produces a consistent quarterly employment measure (`emplvl`) for each observation.

### Aggregation to County-Quarter Level

The dataset contains multiple observations per county and quarter due to different industry subcategories and classifications. To obtain a single value per county and time period, employment will be aggregated by summing across all relevant observations:

- Grouped by: `area_fips`, `year`, `qtr`
- Aggregated variable: `emplvl`

This results in a panel dataset with one observation per county per quarter.

### Removing State-Level Observations

The observation with FIPS code `48000` represents the aggregate for the entire state of Texas rather than an individual county. This row will be removed to ensure consistency at the county level.

### Sorting the Data

The dataset will be sorted by county and time (`area_fips`, `year`, `qtr`) to ensure that observations are in chronological order. This step is necessary for calculating time-based changes.

### Employment Growth

Employment growth will be calculated as the percentage change in employment from the previous quarter within each county. This will be done using a group-by operation on `area_fips`:

- Growth = (current employment − previous employment) / previous employment

This variable (`emplvl_growth`) serves as the primary dependent variable in the analysis, capturing how employment changes over time in response to economic conditions.

In [13]:
qcew["emplvl"] = (
    qcew["month1_emplvl"] +
    qcew["month2_emplvl"] +
    qcew["month3_emplvl"]
) / 3

qcew_clean = qcew.groupby(
    ["area_fips", "year", "qtr"],
    as_index=False
)["emplvl"].sum()

qcew_clean = qcew_clean[qcew_clean["area_fips"] != "48000"]

qcew_clean = qcew_clean.sort_values(["area_fips", "year", "qtr"])

qcew_clean["emplvl_growth"] = qcew_clean.groupby("area_fips")["emplvl"].pct_change()

qcew_clean.head()

,area_fips,year,qtr,emplvl,emplvl_growth
63,48001,2010,1,1698.0,NaN
64,48001,2010,2,1668.0,-0.017668
65,48001,2010,3,1699.0,0.018585
66,48001,2010,4,1773.0,0.043555
67,48001,2011,1,1758.0,-0.008460


In [14]:
# Preview datasets with structure

print("===== QCEW =====")
display(qcew_clean.head())
print("\nQCEW Data Types:")
display(qcew_clean.dtypes)

print("\n===== OIL DATA =====")
display(oil.head())
print("\nOil Data Types:")
display(oil.dtypes)

print("\n===== UNEMPLOYMENT =====")
display(unemp.head())
print("\nUnemployment Data Types:")
display(unemp.dtypes)

===== QCEW =====


,area_fips,year,qtr,emplvl,emplvl_growth
63,48001,2010,1,1698.0,NaN
64,48001,2010,2,1668.0,-0.017668
65,48001,2010,3,1699.0,0.018585
66,48001,2010,4,1773.0,0.043555
67,48001,2011,1,1758.0,-0.008460



QCEW Data Types:


area_fips         object
year               int64
qtr                int64
emplvl           float64
emplvl_growth    float64
dtype: object


===== OIL DATA =====


...

observation_date     object
WTISPLC             float64
dtype: object


===== UNEMPLOYMENT =====


,observation_date,TXUR
0,1976-01-01,5.8
1,1976-02-01,5.8
2,1976-03-01,5.9
3,1976-04-01,5.9
4,1976-05-01,6.0



Unemployment Data Types:


observation_date     object
TXUR                float64
dtype: object

## Preparing Time-Series Data for Merging

The oil price and unemployment datasets are originally recorded at a monthly frequency and require several transformations before they can be merged with the QCEW employment data.

### Datetime Conversion

The `observation_date` variable is initially stored as a string. This will be converted to a datetime format to enable time-based operations such as extracting year and quarter. Without this conversion, it would not be possible to align the datasets across time.

### Extracting Year and Quarter

Since the QCEW dataset is structured at the quarterly level, the oil and unemployment data must be converted to the same time unit. Year and quarter will be extracted from the datetime variable to create consistent merge keys across datasets.

### Aggregation to Quarterly Frequency

The oil price and unemployment datasets contain monthly observations, while the employment data is reported quarterly. To ensure comparability, monthly values will be aggregated into quarterly averages.

This aggregation serves two purposes:
- It aligns the time frequency across datasets
- It smooths short-term fluctuations, providing a more stable measure of economic conditions


In [16]:
# Convert to datetime
oil['observation_date'] = pd.to_datetime(oil['observation_date'])

# Filter to 2010–2025
oil = oil[
    (oil['observation_date'].dt.year >= 2010) &
    (oil['observation_date'].dt.year <= 2025)
]

# Then create year/quarter
oil['year'] = oil['observation_date'].dt.year
oil['qtr'] = oil['observation_date'].dt.quarter

# Aggregate to quarterly
oil_q = oil.groupby(['year', 'qtr'], as_index=False)['WTISPLC'].mean()

In [17]:
# Convert to datetime
unemp['observation_date'] = pd.to_datetime(unemp['observation_date'])

# Filter to 2010–2025
unemp = unemp[
    (unemp['observation_date'].dt.year >= 2010) &
    (unemp['observation_date'].dt.year <= 2025)
]

# Extract year/quarter
unemp['year'] = unemp['observation_date'].dt.year
unemp['qtr'] = unemp['observation_date'].dt.quarter

# Aggregate to quarterly
unemp_q = unemp.groupby(['year', 'qtr'], as_index=False)['TXUR'].mean()

In [19]:
# Preview 
unemp_q.head()

,year,qtr,TXUR
0,2010,1,8.400000
1,2010,2,8.200000
2,2010,3,8.133333
3,2010,4,8.266667
4,2011,1,8.166667


In [20]:
oil_q.head()

,year,qtr,WTISPLC
0,2010,1,78.626667
1,2010,2,77.890000
2,2010,3,76.166667
3,2010,4,85.026667
4,2011,1,93.980000


# 2) Data Merging and Final Dataset Construction

The cleaned employment dataset was merged with oil price and unemployment data using year and quarter as common keys.

All datasets will be aligned to a quarterly frequency to ensure consistency. The resulting dataset contains county-level employment growth alongside macroeconomic variables.

## Merge Strategy

A **left join** was used when merging the datasets, with the QCEW employment data as the base. This ensures that all county-level employment observations are preserved, even if corresponding oil or unemployment data is missing for some periods.

This approach is appropriate because the primary focus of the analysis is on employment outcomes. Oil prices and unemployment rates are used as explanatory variables, so it is important not to drop employment observations due to missing macroeconomic data.

Using other join types would introduce issues:
- An **inner join** would remove any observations where oil or unemployment data is missing, potentially discarding valid employment data.
- A **right join** would prioritize macroeconomic data and could introduce observations without corresponding employment data.
- An **outer join** would include all observations from all datasets, potentially creating many missing values and complicating analysis.

The left join provides a balance by maintaining the integrity of the employment dataset while incorporating relevant macroeconomic variables.

## Merging Steps
1. Left join the QCEW dataset with the Oil dataset into one dataset, "df"
2. Left join the Unemployment dataset with the new "df" dataset
3. Check new dataset for any issues
4. Remove rows without any growth
5. Check data descriptive statistics
6. Replace infinite values with NaN and drop
7. Cap any extreme values to remove noise
8. Save final, cleaned dataset

In [21]:
# Merge datasets

df = qcew_clean.merge(
    oil_q,
    on=['year', 'qtr'],
    how='left'
)

df = df.merge(
    unemp_q,
    on=['year', 'qtr'],
    how='left'
)

df.head()

,area_fips,year,qtr,emplvl,emplvl_growth,WTISPLC,TXUR
0,48001,2010,1,1698.0,NaN,78.626667,8.400000
1,48001,2010,2,1668.0,-0.017668,77.890000,8.200000
2,48001,2010,3,1699.0,0.018585,76.166667,8.133333
3,48001,2010,4,1773.0,0.043555,85.026667,8.266667
4,48001,2011,1,1758.0,-0.008460,93.980000,8.166667


In [22]:
print("Shape:", df.shape)
df.isna().sum()

Final shape: (14482, 7)


area_fips           0
year                0
qtr                 0
emplvl              0
emplvl_growth    4880
WTISPLC             0
TXUR                0
dtype: int64

In [26]:
# Remove rows without growth (first observation per county)
df = df.dropna(subset=['emplvl_growth'])
print("Final shape:", df.shape)
df.isna().sum()

Final shape: (9602, 7)


area_fips        0
year             0
qtr              0
emplvl           0
emplvl_growth    0
WTISPLC          0
TXUR             0
dtype: int64

In [27]:
display(df.describe())

,year,qtr,emplvl,emplvl_growth,WTISPLC,TXUR
count,9602.000000,9602.000000,9602.000000,9602.000000,9602.000000,9602.000000
mean,2017.445636,2.497188,4945.688641,inf,71.597579,5.274148
std,4.419071,1.106948,24030.200092,NaN,20.695921,1.676644
min,2010.000000,1.000000,0.000000,-1.000000,27.806667,3.433333
25%,2014.000000,2.000000,180.000000,-0.065758,55.366667,4.066667
50%,2017.000000,2.000000,886.000000,0.005892,71.836667,4.500000
75%,2021.000000,3.000000,2586.833333,0.070135,92.270000,6.500000
max,2025.000000,4.000000,381564.666667,inf,108.723333,11.533333


In [28]:
# Replace infinite values with NaN (inf comes from a # / 0)
df["emplvl_growth"] = df["emplvl_growth"].replace([np.inf, -np.inf], np.nan)

# Drop those rows (cleanest option for your project)
df = df.dropna(subset=["emplvl_growth"])

# Check again
print(df["emplvl_growth"].describe())

count    9295.000000
mean        0.044967
std         1.057278
min        -1.000000
25%        -0.069462
50%         0.001900
75%         0.062009
max        34.630996
Name: emplvl_growth, dtype: float64


In [29]:
# Cap growth at reasonable bounds
df["emplvl_growth"] = df["emplvl_growth"].clip(lower=-1, upper=1)

# Check again
print(df["emplvl_growth"].describe())

count    9295.000000
mean       -0.023346
std         0.283059
min        -1.000000
25%        -0.069462
50%         0.001900
75%         0.062009
max         1.000000
Name: emplvl_growth, dtype: float64


In [31]:
# FINAL DATASET
df.to_csv("../data_clean/final_dataset.csv", index=False)

## ETL Summary and Final Dataset

This notebook constructed a clean and reproducible dataset by integrating multiple data sources and transforming them into a consistent analytical format.

### Summary of Steps

- Loaded and inspected raw datasets from QCEW, FRED oil prices, and unemployment data
- Addressed data quality issues, including formatting of FIPS codes and datetime conversion
- Filtered QCEW data to include only Texas counties and energy-related industries
- Aggregated employment data to the county-quarter level
- Constructed an employment growth variable to capture changes over time
- Converted oil price and unemployment data from monthly to quarterly frequency
- Restricted all datasets to a relevant time period (2010–2025)
- Merged datasets using a left join to preserve all employment observations

### Final Output

The resulting dataset is a panel dataset at the county-quarter level containing:

- Employment levels and growth rates
- Oil prices (WTI)
- Unemployment rates

This dataset is saved in the `data_clean/` directory.